### VALIDACIÓN RESTO DE POBLACION INGLES

In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [2]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
        SELECT DISTINCT
            a.Identificacion,
            a.Nombre_Estudiante,
            a.Periodo,
            a.Cod_Unidad,
            a.Status,
            a.Nuevo,
            a.promedio,
            a.Semestre_SINU,
            a.ciclo,
            a.SEMESTRE_MEN,
            a.Año,
            a.AÑO_MEN,
            a.[Estado_Alumno],
            a.anio_grado,
            a.Regional,
            a.Programa,
            a.Grado_TC, a.Grado_TG, 
            a.Grado_PR,
            a.pensum,
            c.SNIES,
            c.ESTADO_IDIOMAS,
            c.ESTADO_ACADEMICO,
            c.ESTADO_DEUDA,
            c.ESTADO_GRADO
        FROM
            CUN_REPOSITORIO.Desercion.Resultado_Final_CUN a
        LEFT JOIN
            DBARON.Grados_Saldo c
        ON
                a.Identificacion = c.NUM_IDENTIFICACION
            --AND a.Cod_Unidad = c.COD_UNIDAD
            AND a.Pensum = c.COD_PENSUM
            AND a.ciclo = c.ciclo
        
        ORDER BY
            a.Identificacion,
            a.Periodo;
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [3]:
df.loc[(df['ciclo']=='TECNICO')&(df['Grado_TC']!='-'),'Estado_Graduado'] = 'Graduado'
df.loc[(df['ciclo']=='TECNOLOGO')&(df['Grado_TG']!='-'),'Estado_Graduado'] = 'Graduado'
df.loc[(df['ciclo']=='PROFESIONAL')&(df['Grado_PR']!='-'),'Estado_Graduado'] = 'Graduado'

In [4]:
# Crear una marca booleana de si ya está graduado o no
df['Graduado_Marca'] = df['Estado_Graduado'] == 'Graduado'

# Agrupar por estudiante y ciclo, y verificar si alguno está Graduado
df['Graduado_Final'] = df.groupby(['Identificacion', 'ciclo'])['Graduado_Marca'].transform('max')

# Reemplazar True/False por 'Graduado' o mantener NaN
df['Estado_Graduado'] = df['Graduado_Final'].apply(lambda x: 'Graduado' if x else None)

# Eliminar columna temporal si no la necesitas
df.drop(columns=['Graduado_Marca', 'Graduado_Final'], inplace=True)


In [5]:
df = df[df['Estado_Graduado']!='Graduado']

In [6]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT DISTINCT * FROM SRUBIO.GRADO_OPORTUNO2
            
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_grado_oportuno = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [7]:
df_grado_oportuno = df_grado_oportuno[['Identificacion', 'Periodo', 'ciclo', 'Semestre_SINU', 'pensum',
       'Cod_Unidad', 'AÑO_GRAD_OPORTUNA', 'SEMESTRE_MEN_GRAD_OPORTUNA',
       'Programa', 'Regional', 'ESTADO_ACADEMICO', 'ESTADO_IDIOMAS',
       'Tipo_grado', 'OPCION_GRADO', 'INGLES', 
       'PENSAMIENTO CUNISTA', 'PRACTICAS', 'NUMERO',
       'ESPECIALIZACIÓN', 'PROFESIONAL', 'TECNÓLOGO', 'TÉCNICO', 'TIPO',
       'Clasificacion_Final', 'GRADUADO', 'Ultimo_periodo']]

In [8]:
df_grado_oportuno['llave'] = df_grado_oportuno['Identificacion'] 
df['llave'] = df['Identificacion']

df2=df[~df['llave'].isin(df_grado_oportuno['llave'])]

In [9]:
df2.shape

(341182, 27)

In [10]:
df_grado_oportuno.shape

(94827, 27)

In [11]:
df = pd.concat([df2,df_grado_oportuno],ignore_index=True)

In [12]:
df.shape

(436009, 43)

In [13]:
a=df[df['Periodo']=='2025A']
a['Identificacion'].nunique()

9694

In [14]:
# df = df[~df['Status'].isin(['Desercion CUN','Retiro'])]

### INGLES DENTRO DE MALLA

In [15]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
                        
            
                SELECT * 
                FROM OPENQUERY([172.16.1.175], 
                    '
                    SELECT 
                        b.NUM_IDENTIFICACION, 
                        a.COD_MATERIA
                    FROM 
                        SINU.SRC_HIS_ACADEMICA a
                        LEFT JOIN SINU.SRC_ALUM_PROGRAMA c ON a.ID_ALUM_PROGRAMA = c.ID_ALUM_PROGRAMA
                        LEFT JOIN SINU.BAS_TERCERO b ON b.ID_TERCERO = c.ID_TERCERO
                    WHERE 
                        a.COD_MATERIA IN (''A1I00'', ''A2I00'', ''B1I00'', ''564I6'', ''564I7'', ''564I8'', 
                                            ''564I9'', ''564I1'', ''564I2'', ''564I3'', ''564I4'', ''564I5'', 
                                            ''56N10'', ''56N11'', ''56N12'', ''56N13'', ''A1I00'', ''56I10'') 
                        AND a.IND_APROBADA = ''1''
                        
                    '
                );
            
    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_dm = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [16]:
df_dm = df_dm.drop_duplicates()

In [17]:
df_dm_conteo = df_dm.groupby('NUM_IDENTIFICACION')['COD_MATERIA'].count().reset_index()

In [18]:
df_dm_conteo['TIPO_INGLES_MALLA'] = 'INGLES INCLUIDO EN MALLA'

In [19]:
df_dm_conteo.rename(columns={'COD_MATERIA':'NIVELESAPROBADOS'},inplace=True)

### INGLES FUERA DE MALLA

In [20]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
                    SELECT * 
            FROM OPENQUERY([172.16.1.175], 
                '
                SELECT 
                    B.NUM_IDENTIFICACION, 
                    SUM(a.IND_APROBADA) AS NivelesAprobados
                FROM 
                    SINU.SRC_HIS_ACADEMICA a
                    LEFT JOIN SINU.SRC_ALUM_PROGRAMA c ON a.ID_ALUM_PROGRAMA = c.ID_ALUM_PROGRAMA
                    LEFT JOIN SINU.BAS_TERCERO b ON b.ID_TERCERO = c.ID_TERCERO
                WHERE 
                    (a.COD_PERIODO LIKE ''%BE%'' OR a.COD_PERIODO LIKE ''%I%'' OR a.COD_PERIODO LIKE ''%M%'') 
                    AND a.IND_APROBADA = ''1''
                GROUP BY 
                    B.NUM_IDENTIFICACION
                '
            )
    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_ing = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [21]:
df_ing['TIPO_INGLES_MALLA'] = 'FUERA DE MALLA'
df_ing['NUM_IDENTIFICACION'] = df_ing['NUM_IDENTIFICACION'].astype(str)
df_ing['NIVELESAPROBADOS'] = df_ing['NIVELESAPROBADOS'].astype(int)

In [22]:
df_ing2 = pd.concat([df_ing,df_dm_conteo],ignore_index=True)

In [23]:
# Ordenar por NUM_IDENTIFICACION y luego por NIVELES_APROBADOS en orden descendente
df_ing2 = df_ing2.sort_values(by=['NUM_IDENTIFICACION', 'NIVELESAPROBADOS'], ascending=[True, False])

# Eliminar duplicados, manteniendo solo el registro con mayor número de niveles aprobados
df_ing2 = df_ing2.drop_duplicates(subset=['NUM_IDENTIFICACION'], keep='first')

In [24]:
df_ing2.columns

Index(['NUM_IDENTIFICACION', 'NIVELESAPROBADOS', 'TIPO_INGLES_MALLA'], dtype='object')

In [25]:
df.columns

Index(['Identificacion', 'Nombre_Estudiante', 'Periodo', 'Cod_Unidad',
       'Status', 'Nuevo', 'promedio', 'Semestre_SINU', 'ciclo', 'SEMESTRE_MEN',
       'Año', 'AÑO_MEN', 'Estado_Alumno', 'anio_grado', 'Regional', 'Programa',
       'Grado_TC', 'Grado_TG', 'Grado_PR', 'pensum', 'SNIES', 'ESTADO_IDIOMAS',
       'ESTADO_ACADEMICO', 'ESTADO_DEUDA', 'ESTADO_GRADO', 'Estado_Graduado',
       'llave', 'AÑO_GRAD_OPORTUNA', 'SEMESTRE_MEN_GRAD_OPORTUNA',
       'Tipo_grado', 'OPCION_GRADO', 'INGLES', 'PENSAMIENTO CUNISTA',
       'PRACTICAS', 'NUMERO', 'ESPECIALIZACIÓN', 'PROFESIONAL', 'TECNÓLOGO',
       'TÉCNICO', 'TIPO', 'Clasificacion_Final', 'GRADUADO', 'Ultimo_periodo'],
      dtype='object')

In [26]:
result = df.merge(
    df_ing2[['NUM_IDENTIFICACION','NIVELESAPROBADOS','TIPO_INGLES_MALLA']],
    left_on='Identificacion',
    right_on='NUM_IDENTIFICACION',
    how='left'
)

In [27]:
result.columns

Index(['Identificacion', 'Nombre_Estudiante', 'Periodo', 'Cod_Unidad',
       'Status', 'Nuevo', 'promedio', 'Semestre_SINU', 'ciclo', 'SEMESTRE_MEN',
       'Año', 'AÑO_MEN', 'Estado_Alumno', 'anio_grado', 'Regional', 'Programa',
       'Grado_TC', 'Grado_TG', 'Grado_PR', 'pensum', 'SNIES', 'ESTADO_IDIOMAS',
       'ESTADO_ACADEMICO', 'ESTADO_DEUDA', 'ESTADO_GRADO', 'Estado_Graduado',
       'llave', 'AÑO_GRAD_OPORTUNA', 'SEMESTRE_MEN_GRAD_OPORTUNA',
       'Tipo_grado', 'OPCION_GRADO', 'INGLES', 'PENSAMIENTO CUNISTA',
       'PRACTICAS', 'NUMERO', 'ESPECIALIZACIÓN', 'PROFESIONAL', 'TECNÓLOGO',
       'TÉCNICO', 'TIPO', 'Clasificacion_Final', 'GRADUADO', 'Ultimo_periodo',
       'NUM_IDENTIFICACION', 'NIVELESAPROBADOS', 'TIPO_INGLES_MALLA'],
      dtype='object')

In [28]:
import numpy as np

result['INGLES'] = np.where(
    result['TIPO_INGLES_MALLA'] == 'FUERA DE MALLA',
    np.where(
        (result['ciclo'] == 'TECNICO') & (result['NIVELESAPROBADOS'] >= 3), 'APROBÓ',
        np.where(
            (result['ciclo'] == 'TECNOLOGO') & (result['NIVELESAPROBADOS'] >= 6), 'APROBÓ',
            np.where(
                (result['ciclo'] == 'PROFESIONAL') & (result['NIVELESAPROBADOS'] >= 9), 'APROBÓ',
                'NO CUMPLE'  # Se mantiene 'NO CUMPLE' solo para los casos dentro de 'FUERA DE MALLA'
            )
        )
    ),
    ''
)

In [29]:
result['pensum'] = result['pensum'].str.rstrip('ND')

#### SE VALIDA DE LOS QUE NO CUMPLEN CUALES ESTAN EN PERIODOS ACTIVOS O EN CONVOCATORIA INGLES SE DEBE CORRER Periodos_ingles.ipynb

In [30]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT DISTINCT NUM_IDENTIFICACION, Periodo AS Periodo_conv, Estado,Niveles as Cursando_niveles  
            FROM SRUBIO.TABLA_CONVOCATORIA_INGLES
    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_conv_act = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [31]:
result = pd.merge(result,df_conv_act[['NUM_IDENTIFICACION','Periodo_conv','Estado','Cursando_niveles']],how='left',left_on=['Identificacion'],right_on=['NUM_IDENTIFICACION'])

In [32]:
result.columns

Index(['Identificacion', 'Nombre_Estudiante', 'Periodo', 'Cod_Unidad',
       'Status', 'Nuevo', 'promedio', 'Semestre_SINU', 'ciclo', 'SEMESTRE_MEN',
       'Año', 'AÑO_MEN', 'Estado_Alumno', 'anio_grado', 'Regional', 'Programa',
       'Grado_TC', 'Grado_TG', 'Grado_PR', 'pensum', 'SNIES', 'ESTADO_IDIOMAS',
       'ESTADO_ACADEMICO', 'ESTADO_DEUDA', 'ESTADO_GRADO', 'Estado_Graduado',
       'llave', 'AÑO_GRAD_OPORTUNA', 'SEMESTRE_MEN_GRAD_OPORTUNA',
       'Tipo_grado', 'OPCION_GRADO', 'INGLES', 'PENSAMIENTO CUNISTA',
       'PRACTICAS', 'NUMERO', 'ESPECIALIZACIÓN', 'PROFESIONAL', 'TECNÓLOGO',
       'TÉCNICO', 'TIPO', 'Clasificacion_Final', 'GRADUADO', 'Ultimo_periodo',
       'NUM_IDENTIFICACION_x', 'NIVELESAPROBADOS', 'TIPO_INGLES_MALLA',
       'NUM_IDENTIFICACION_y', 'Periodo_conv', 'Estado', 'Cursando_niveles'],
      dtype='object')

In [33]:
result.loc[(result['INGLES'].isin(['']))&(result['Estado'].notna()),'INGLES'] = result['Estado']

In [34]:
# Diccionario con las reglas por pensum y ciclo
requisitos_ingles = {
    '8014': {'TECNICO': 2, 'TECNOLOGO': 3, 'PROFESIONAL': 7},
    '3029': {'TECNICO': 2, 'TECNOLOGO': 4, 'PROFESIONAL': 5},
    'VG001': {'TECNICO': 1, 'TECNOLOGO': 2, 'PROFESIONAL': 5},
    'default': {'TECNICO': 3, 'TECNOLOGO': 6, 'PROFESIONAL': 9}  # Reglas generales
}

# Función para evaluar la aprobación
def evaluar_aprobacion(row):
    if row['TIPO_INGLES_MALLA'] != 'INGLES INCLUIDO EN MALLA':
        return row.get('INGLES', '')  # Mantiene la marca actual si ya tiene un valor
    
    # Obtiene las reglas según el pensum, usa 'default' si no está en el diccionario
    reglas = requisitos_ingles.get(row['pensum'], requisitos_ingles['default'])
    
    # Evalúa si cumple con el mínimo de niveles requeridos
    return 'APROBÓ' if row['NIVELESAPROBADOS'] >= reglas.get(row['ciclo'], 0) else 'NO CUMPLE'


# Aplica la función a cada fila del DataFrame
result['INGLES'] = result.apply(evaluar_aprobacion, axis=1)

In [35]:
result.loc[result['pensum'].isin(['8014','3029','VG001']) & (result['NIVELESAPROBADOS'].isna()),'INGLES'] = 'NO CUMPLE'
result.loc[result['TIPO_INGLES_MALLA'].isna(),'TIPO_INGLES_MALLA'] = 'FUERA DE MALLA'

In [36]:
result.loc[(result['NIVELESAPROBADOS'].isna())&(result['INGLES']==''),'INGLES'] = 'NO CUMPLE'

In [37]:
result2 = result[~result['Status'].isin(['Desercion CUN','Retiro'])]
result2 = result[result['Identificacion']!='None']

In [38]:
result2.loc[(result2['INGLES']=='NO CUMPLE')&(result2['Estado'].notna()),'INGLES'] = result2['Estado']

In [39]:
def asignar_niveles(row):
    if row['pensum'] in requisitos_ingles:
        return requisitos_ingles[row['pensum']].get(row['ciclo'], None)
    else:
        return {'TECNICO': 3, 'TECNOLOGO': 6, 'PROFESIONAL': 9}.get(row['ciclo'], None)

# Aplicar la función al DataFrame
result2['Niveles_requeridos'] = result2.apply(asignar_niveles, axis=1)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_17480\1947897974.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result2['Niveles_requeridos'] = result2.apply(asignar_niveles, axis=1)


In [40]:
result2['NIVELESAPROBADOS'].fillna(0, inplace=True)
result2['Cursando_niveles'].fillna(0, inplace=True)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_17480\2862257665.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result2['NIVELESAPROBADOS'].fillna(0, inplace=True)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_17480\2862257665.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result2['NIVELESAPROBADOS'].fillna(0, i

In [41]:
result2['Niveles_faltantes'] = result2['Niveles_requeridos']-(result2['NIVELESAPROBADOS'] + result2['Cursando_niveles'])

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_17480\3898191072.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result2['Niveles_faltantes'] = result2['Niveles_requeridos']-(result2['NIVELESAPROBADOS'] + result2['Cursando_niveles'])


In [42]:
# Aplicar la condición para Niveles_faltantes
result2.loc[result2['Niveles_faltantes'] < 0, 'Niveles_faltantes'] = 0

# Aplicar la condición para Niveles_aprobados
result2.loc[result2['NIVELESAPROBADOS'] > 9, 'NIVELESAPROBADOS'] = 9

In [43]:
result2.loc[result2['INGLES']=='APROBÓ','Estado_Bi'] = 'Nivelado'
result2.loc[result2['INGLES']=='NO CUMPLE','Estado_Bi'] = 'Niveles Pendientes'
result2.loc[result2['INGLES']=='PERIODO ACTIVO','Estado_Bi'] = 'En Periodo Activo'
result2.loc[result2['INGLES']=='EN CONVOCATORIA','Estado_Bi'] = 'En convocatoria'

# result2.loc[(result2['TIPO_INGLES_MALLA']=='INGLES INCLUIDO EN MALLA')&
#             (result2['Estado_Bi'].isna()),'Estado_Bi'] = 'Ingles Inmerso en malla'

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_17480\139698378.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result2.loc[result2['INGLES']=='APROBÓ','Estado_Bi'] = 'Nivelado'


In [44]:
result2['Semestre_SINU'].dtype

dtype('O')

In [45]:
result2['Semestre_SINU'] = result2['Semestre_SINU'].fillna(0).astype('int64')

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_17480\1985641144.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result2['Semestre_SINU'] = result2['Semestre_SINU'].fillna(0).astype('int64')


In [46]:
result2 = result2.sort_values(by=['Semestre_SINU','Periodo'])

In [47]:
ultimo_periodo = result2.groupby(['Identificacion', 'ciclo']).last().reset_index()

In [48]:
a=ultimo_periodo[(ultimo_periodo['Periodo']=='25V01')&(ultimo_periodo['INGLES']=='NO CUMPLE')]
a['Identificacion'].nunique()

16156

In [49]:
a=ultimo_periodo[(ultimo_periodo['Periodo']=='2025A')]
a['Identificacion'].nunique()

9692

In [52]:
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine

host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
schema_name = 'SRUBIO'
table_name = 'MARCAS_INGLES'


url = f'mssql+pyodbc://@{host}:{port}/{database}?driver=ODBC Driver 17 for SQL Server'
engine = create_engine(url, fast_executemany=True)

fecha_corte_actual = datetime.today().strftime('%Y-%m')
ultimo_periodo['FECHA_CORTE'] = fecha_corte_actual

with engine.connect() as conn:
    table_exists = pd.read_sql(f"""
        SELECT * FROM INFORMATION_SCHEMA.TABLES 
        WHERE TABLE_SCHEMA = '{schema_name}' AND TABLE_NAME = '{table_name}'
    """, conn)

if not table_exists.empty:
    with engine.begin() as conn:
        conn.execute(
            text(f"""
                DELETE FROM [{schema_name}].[{table_name}]
                WHERE FECHA_CORTE = :fecha
            """),
            {"fecha": fecha_corte_actual}
        )

ultimo_periodo.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize=10000
)

engine.dispose()


In [ ]:
# # Connection to database
# host = '172.16.1.33'
# port = '1433'
# database = 'CUN_REPOSITORIO'
# # username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# # password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
# #url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
# url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
# engine = create_engine(url, fast_executemany=True)

# table_name = 'MARCAS_INGLES'
# schema_name = 'SRUBIO'
# # Send the data to sql server
# ultimo_periodo.to_sql(
#     name=table_name,
#     schema=schema_name,
#     con=engine,
#     if_exists='replace',
#     index=False,
#     chunksize = 10000
# )

# # End connection
# engine.dispose()

In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT DISTINCT * FROM SRUBIO.GRADO_OPORTUNO2
    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_grad_opor = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [ ]:
a=df_grad_opor[(df_grad_opor['Periodo']=='2025A')]
a['Identificacion'].nunique()

9431

In [ ]:
df_grad_opor[df_grad_opor['Identificacion']=='73229215']

,Identificacion,Periodo,ciclo,Semestre_SINU,pensum,Cod_Unidad,AÑO_GRAD_OPORTUNA,SEMESTRE_MEN_GRAD_OPORTUNA,Programa,Regional,...,PRACTICAS,NUMERO,ESPECIALIZACION,PROFESIONAL,TECNICO,TECNOLOGO,TIPO,Clasificacion_Final,GRADUADO,Ultimo_periodo
